# Занятие 2. Практика: NumPy, скорость и первый анализ данных

На прошлом занятии мы познакомились с ноутбуком, NumPy и Pandas.
Сегодня:
1. сравним скорость NumPy и обычных списков на миллионе чисел;
2. посчитаем агрегаты без циклов;
3. прочитаем настоящие данные (CSV и Excel);
4. сделаем первичный осмотр таблицы (`.info()`, `.describe()`);
5. научимся выбирать строки и столбцы через `.loc` и `.iloc`.

Датасет — подготовка **1000 школьников к ЕГЭ**.


In [ ]:
# импортируем библиотеки
import numpy as np
import pandas as pd

---
## Часть 1. NumPy против списков: кто быстрее?

Возьмём **миллион** чисел и сложим их двумя способами: обычным списком и массивом NumPy.
Замерим время волшебной командой `%timeit` — она сама несколько раз запускает код
и показывает среднее время.


In [ ]:
N = 1_000_000
py_list = list(range(N))       # обычный список
np_array = np.arange(N)        # массив NumPy
print('Готово. В каждом по', N, 'чисел.')

### Что такое `%timeit`

`%timeit` — это «секундомер» ноутбука. Он берёт вашу строку кода и запускает её **много раз**,
а потом показывает, **сколько в среднем** времени она занимает. Так измерение получается точным,
а не случайным.

Пишется просто: `%timeit` и дальше код, время которого измеряем. Например, `%timeit sum(py_list)`.

**Как читать результат.** После запуска появится строка примерно такого вида:

```
156 µs ± 3.18 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
```

Разберём её по частям:

| Кусок | Что значит простыми словами |
|---|---|
| `156 µs` | **среднее время одного запуска** — 156 микросекунд |
| `± 3.18 µs` | насколько результат «прыгает» от запуска к запуску (разброс) |
| `per loop` | «за один прогон» кода |
| `7 runs` | измерение повторили 7 раз, чтобы быть точнее |
| `10,000 loops each` | в каждом измерении код прогнали 10 000 раз |

> **µs — это микросекунда**, одна миллионная доля секунды (0.000001 с).
> Бывают ещё `ms` — миллисекунды (тысячная секунды) и `s` — секунды.
> Порядок по возрастанию: **µs → ms → s**.

**Главное правило:** чем **меньше** число перед `µs`/`ms`/`s`, тем **быстрее** код.
Дальше мы сравним время у списка и у массива NumPy — и увидим разницу своими глазами.

### Сумма чисел


In [ ]:
# Способ 1: сумма списка через встроенный sum()
%timeit sum(py_list)

In [ ]:
# Способ 2: сумма массива через NumPy
%timeit np_array.sum()

Сравните числа: у NumPy время в **десятки раз меньше**. Причина — векторизация:
NumPy складывает всё сразу, а не по одному элементу в цикле.

### Умножение каждого числа на 2


In [ ]:
# Список: нужен цикл (списковое включение)
%timeit [x * 2 for x in py_list]

In [ ]:
# Массив: одна операция сразу над всеми
%timeit np_array * 2

**Вывод:** для вычислений с числами массив NumPy намного быстрее и короче в записи.
Именно поэтому Pandas построен на NumPy.


---
## Часть 2. Агрегаты без циклов

На массиве легко считать статистику одной командой.


In [ ]:
data = np.array([90, 75, 60, 82, 95, 70, 88, 64])

print('сумма   :', np.sum(data))
print('среднее :', np.mean(data))
print('минимум :', np.min(data))
print('максимум:', np.max(data))

Эти же функции работают и на больших массивах мгновенно.


In [ ]:
big_array = np.arange(1, N + 1)     # числа от 1 до миллиона
print('сумма миллиона чисел:', big_array.sum())
print('среднее:', big_array.mean())

---
## Часть 3. Читаем настоящие данные

Данные обычно лежат в файлах **CSV** или **Excel (XLSX)**. Pandas читает оба формата.

### Что такое CSV

**CSV** (*Comma-Separated Values* — «значения, разделённые запятыми») — это обычный
**текстовый файл**, где таблица записана строками, а ячейки в строке разделены
каким-то символом-разделителем (запятой `,` или точкой с запятой `;`). Первая строка
обычно содержит названия столбцов. Открыть такой файл можно даже в «Блокноте» —
внутри просто текст:

```
имя;город;балл_егэ
Роман;Нижний Новгород;47
Арина;Ростов-на-Дону;66
```

### Чем CSV лучше XLSX

XLSX — это формат Excel: сложный архив со стилями, формулами, вкладками и оформлением.
Для хранения **данных** CSV чаще удобнее:

| | CSV | XLSX (Excel) |
|---|---|---|
| **Что внутри** | простой текст | сложный архив (стили, формулы, вкладки) |
| **Размер** | маленький (только данные) | больше (тащит с собой оформление) |
| **Скорость чтения** | быстро | медленнее |
| **Кто откроет** | любая программа, любой язык | нужен Excel или спец. библиотека |
| **Годится для программ** | да, это стандарт для обмена данными | не всегда |

Коротко: **CSV — универсальный, лёгкий и быстрый** формат для данных, поэтому его
используют почти везде. XLSX удобен, когда нужны формулы, графики и красивое оформление
внутри самого файла, но как «хранилище чисел» он тяжелее и медленнее.

### Чем читаем таблицы: метод `read_...`

Для чтения таблиц в Pandas используют семейство методов **`read_...`**:
`pd.read_csv(...)` — для CSV, `pd.read_excel(...)` — для Excel. Достаточно указать
путь к файлу — и таблица загрузится в `DataFrame`.

У этих методов **много параметров**, которые управляют чтением. Например, `sep`
задаёт символ-разделитель в CSV (у нас `sep=';'`). Есть и другие: сколько строк
пропустить, какую строку считать заголовком, какие столбцы взять и так далее.
Пока запомним только то, что нужно сейчас, — **с остальными параметрами мы
подробнее познакомимся в будущем**.


In [ ]:
# CSV: разделитель в нашем файле — точка с запятой (sep=';')
df = pd.read_csv('data/ege_students.csv', sep=';')
df.head()

In [ ]:
# То же самое можно прочитать из Excel:
df_excel = pd.read_excel('data/ege_students.xlsx')
df_excel.head()

---
## Часть 4. Первичный осмотр таблицы

Прежде чем что-то анализировать, таблицу всегда **осматривают**. Три главных инструмента:
- **`.head()`** — первые строки (по умолчанию 5);
- **`.info()`** — сводка: столбцы, типы, пропуски;
- **`.describe()`** — статистика по числовым столбцам.


In [ ]:
# Первые 5 строк
df.head()

In [ ]:
# Сводка: сколько строк, какие типы, где пропуски
df.info()

В выводе `.info()` смотрите на два момента:
- **Non-Null Count** — сколько заполненных значений в столбце;
- **Dtype** — тип столбца (число `int/float` или текст `object`).


In [ ]:
# Статистика по числовым столбцам: среднее, минимум, максимум и др.
df.describe().round(1)

`.describe()` сразу отвечает на вопросы: какой средний балл ЕГЭ, минимальный и максимальный,
сколько в среднем часов готовятся ученики и т. д.


In [ ]:
# Отдельно посчитаем главные агрегаты по баллу ЕГЭ
print('Средний балл ЕГЭ :', df['балл_егэ'].mean().round(1))
print('Максимальный балл:', df['балл_егэ'].max())
print('Минимальный балл :', df['балл_егэ'].min())

---
## Часть 5. Индексация: как достать нужные строки и столбцы

Таблица `df` — это как большая таблица в Excel. Часто нам нужна не вся она,
а только кусочек: один столбец, одна строка, несколько строк подряд или
конкретная «клетка». Разберём все способы по порядку.

### 5.1. Обращение к столбцу через квадратные скобки

Самый частый приём — взять **один столбец по имени** в квадратных скобках `df['имя_столбца']`.
Получится «столбик» значений (объект `Series`).


In [ ]:
# один столбец по названию
df['балл_егэ'].head()

### 5.2. Обращение к столбцу через точку

Если в названии столбца **нет пробелов и спецсимволов**, к нему можно обратиться
короче — через точку: `df.столбец`. Это то же самое, что и квадратные скобки.

> ⚠️ Через точку **нельзя**, если в имени есть пробел (например, `df.любимый предмет`
> не сработает — тут только `df['любимый предмет']`). Квадратные скобки работают всегда.


In [ ]:
# то же самое через точку (пробелов в имени нет — можно)
df.балл_егэ.head()

### 5.3. Несколько столбцов сразу

Чтобы взять **несколько столбцов**, передаём **список имён** — обратите внимание на
**двойные скобки** `[[ ... ]]`: внешние — это индексация, внутренние — список.


In [ ]:
# несколько столбцов: список имён в двойных скобках
df[['имя', 'город', 'балл_егэ']].head()

### 5.4. Два инструмента для строк и клеток: `.loc` и `.iloc`

Когда нужны **строки** (или конкретные клетки), используют два инструмента:

| Инструмент | Как обращаемся | Пример |
|---|---|---|
| **`.loc`** | по **названиям** (метка строки, имя столбца) | `df.loc[0, 'имя']` |
| **`.iloc`** | по **номерам-позициям** (счёт с 0) | `df.iloc[0, 1]` |

Формат у обоих одинаковый: `df.loc[строки, столбцы]` и `df.iloc[строки, столбцы]`.
У нашей таблицы метки строк — это числа `0, 1, 2, ...`, поэтому для строк
`.loc` и `.iloc` часто выглядят похоже, но это **разные вещи** (см. про срезы ниже).

### 5.5. Одна конкретная строка

`.loc[метка]` — строка по её метке, `.iloc[номер]` — строка по её позиции.


In [ ]:
# строка с меткой 0 (первый ученик)
df.loc[0]

In [ ]:
# та же строка по позиции (нумерация с нуля)
df.iloc[0]

### 5.6. Одна конкретная клетка (строка + столбец)

Указываем и строку, и столбец. Через `.loc` — по именам, через `.iloc` — по номерам.


In [ ]:
# клетка: строка 0, столбец 'имя'
print('По названию :', df.loc[0, 'имя'])

# та же клетка по номерам: строка 0, столбец 1 ('имя' — второй столбец, счёт с 0)
print('По номеру   :', df.iloc[0, 1])

### 5.7. Один столбец через `.loc` / `.iloc`

Если поставить `:` на месте строк — значит «все строки».


In [ ]:
# все строки, столбец 'город' — первые 5
df.loc[:, 'город'].head()

In [ ]:
# все строки, столбец с номером 4 ('город') — первые 5
df.iloc[:, 4].head()

### 5.8. Срезы: несколько строк подряд

Срез — это диапазон «от и до». Здесь у `.loc` и `.iloc` есть **важное отличие**:

- **`.loc[3:6]`** — по меткам, **правая граница ВКЛЮЧАЕТСЯ** → строки 3, 4, 5, **6**;
- **`.iloc[3:6]`** — по позициям, как в обычном Python, **правая граница НЕ включается**
  → строки 3, 4, 5 (без 6).

Запомнить легко: `.loc` — «по-человечески, до и включая», `.iloc` — «как в Python, до, но не включая».


In [ ]:
# .loc[3:6] — включает строку 6 → всего 4 строки
df.loc[3:6, ['имя', 'балл_егэ']]

In [ ]:
# .iloc[3:6] — НЕ включает строку 6 → всего 3 строки
df.iloc[3:6, [1, 14]]

### 5.9. Срез от начала и до конца

Границу можно не писать: пусто слева — «с самого начала», пусто справа — «до самого конца».


In [ ]:
# первые 5 строк: от начала до позиции 5 (не включая)
df.iloc[:5]

In [ ]:
# последние 5 строк: с позиции -5 и до конца
df.iloc[-5:]

In [ ]:
# все строки, но только столбцы с 1-го по 4-й (по номерам)
df.iloc[:, 1:5].head()

---
## Практика

Ниже — 12 заданий на изученный материал. В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен;
- **выведите результат** через `print()` или последней строкой.

В некоторых заданиях есть пункт **Вывод:** — там нужно короткое предложение о том, что получилось и как это трактовать.

Работаем с таблицей `df`, загруженной выше. Столбцы:
`ученик_id`, `имя`, `пол`, `возраст`, `город`, `тип_школы`,
`любимый_предмет`, `часов_подготовки_в_неделю`, `репетитор`, `кол_во_пробников`,
`пропусков_уроков`, `мотивация_1_10`, `часов_сна`, `средний_балл_года`, `балл_егэ`, `сдал`.

**Максимум за практику — 30 баллов.**


### <font color='DarkOrange'>Задание 1 [1 балл]</font>

Определите размер таблицы `df` — сколько в ней **строк** и **столбцов**.


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Найдите **имя ученика** в строке с меткой **500**. Используйте `.loc`.


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 3 [2 балла]</font>

Найдите **максимальный** и **минимальный** балл ЕГЭ в таблице.


In [ ]:
# Ваш код


**Вывод:** *одним предложением — каков разрыв между максимумом и минимумом и о чём это говорит.*


### <font color='DarkOrange'>Задание 4 [2 балла]</font>

Посчитайте **средний балл ЕГЭ** по всем ученикам и **округлите до одного знака** после запятой.


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Какой **любимый предмет** у ученика в строке с меткой **100**? Используйте `.loc`.


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 6 [3 балла]</font>

Из какого **города** первый ученик таблицы (строка с позицией 0)?
Достаньте значение через **`.iloc`** — по номерам строки и столбца.


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 7 [2 балла]</font>

Сколько часов в неделю готовился ученик с **самой долгой подготовкой**?


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 8 [3 балла]</font>

Возьмите **последние 100 учеников** таблицы и посчитайте у них **средний балл ЕГЭ**
(округлите до одного знака после запятой).


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 9 [3 балла]</font>

Среди **первых 250 учеников** найдите **максимальный возраст**.


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 10 [3 балла]</font>

Возьмите учеников **с позиции 300 по 399** (сто человек) и посчитайте их
**среднее число часов сна** (округлите до одного знака после запятой).


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 11 [3 балла]</font>

Возьмите учеников **с позиции 100 по 249** (150 человек) и посчитайте их
**среднее количество пробников** (столбец `кол_во_пробников`, округлите до одного знака после запятой).


In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 12 [4 балла]</font>

Возьмите строки **с меткой 500 по 600 включительно** (через `.loc`) и посчитайте
их **средний балл за год** (столбец `средний_балл_года`, округлите до двух знаков).


In [ ]:
# Ваш код


**Вывод:** *одним предложением — сколько строк попало в срез и какой средний балл получился.*
